In [99]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd
from sklearn.metrics import mean_squared_error

from sklearn.preprocessing import OneHotEncoder


In [100]:
# Preprocessing : changed sex and Embarked to numerical value 

from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

data = pd.read_csv('./train.csv')
data[['Age', 'Fare']] = scaler.fit_transform(data[['Age', 'Fare']])
data.drop(['PassengerId', 'Ticket', 'Cabin', 'Name'], axis=1, inplace=True)
data['Age'].fillna(data['Age'].median(), inplace=True)
data['Fare'].fillna(data['Fare'].median(), inplace=True)  # if any missing

C:\Users\user\AppData\Local\Temp\ipykernel_17596\2661827017.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Age'].fillna(data['Age'].median(), inplace=True)
C:\Users\user\AppData\Local\Temp\ipykernel_17596\2661827017.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

Fo

In [101]:

# Encoding train data

oneHotEncoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded = oneHotEncoder.fit_transform(data[['Embarked']])
encoded_cols = oneHotEncoder.get_feature_names_out(['Embarked'])
encoded_df = pd.DataFrame(encoded, columns=encoded_cols, index=data.index)
data = pd.concat([data.drop(columns=['Embarked']), encoded_df], axis=1)
data['Sex'] = data['Sex'].map({'male': 1, 'female': 0})
data.drop(['Survived'], axis=1, inplace=True)


# Encoding test data
test_data = pd.read_csv("./test.csv")
test_data['Age'].fillna(data['Age'].median(), inplace=True)  # use median from training
test_data['Fare'].fillna(data['Fare'].median(), inplace=True)  # if any missing
test_data['Sex'] = test_data['Sex'].map({'male': 1, 'female': 0})

test_embarked = oneHotEncoder.transform(test_data[['Embarked']])
encoded_cols = oneHotEncoder.get_feature_names_out(['Embarked'])
test_embarked_df = pd.DataFrame(test_embarked, columns=encoded_cols, index=test_data.index)
test_data = pd.concat([test_data.drop(columns=['Embarked']), test_embarked_df], axis=1)
test_data[['Age', 'Fare']] = scaler.transform(test_data[['Age', 'Fare']])

y = data['Survived']



C:\Users\user\AppData\Local\Temp\ipykernel_17596\1169961808.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_data['Age'].fillna(data['Age'].median(), inplace=True)  # use median from training
C:\Users\user\AppData\Local\Temp\ipykernel_17596\1169961808.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting va

KeyError: 'Survived'

In [ ]:
X_train, x_valid, y_train, y_valid = train_test_split(data, y, test_size=0.3, random_state=39)

In [ ]:
randomforest_model = RandomForestClassifier()

param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid_search = GridSearchCV(
  estimator=randomforest_model,
  cv=3,
  n_jobs=-1,
  param_grid=param_grid
)

grid_search.fit(X_train, y_train)

In [ ]:
mse = mean_squared_error(y_valid, grid_search.predict(x_valid))


In [ ]:
predictions = grid_search.predict(test_data)
